# FIREWATCH AI — Coverage-Cautious Activity Calendar

**Stage 06.** Build daily counts from raw observation rows while distinguishing dates with detections from dates with no rows and unknown coverage. Compare equivalent periods descriptively. This stage does not label anomalies or infer zero fire activity.

Change `YEAR` and `MONTH` to inspect a different calendar month.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_root(start=Path.cwd()):
    for p in [start,*start.parents]:
        if (p/'2025').is_dir() and (p/'2026').is_dir(): return p
    raise FileNotFoundError('Cannot locate project root')
ROOT=find_root(); sys.path.insert(0,str(ROOT))
from src.firewatch.analytics import build_daily_activity_calendar,summarize_calendar
DATA=ROOT/'data'/'processed'/'observations.parquet'
YEAR=2025; MONTH=10
columns=['year_folder','product_code','satellite','stream_type','source_file','observation_datetime_utc','validation_status']
observations=pd.read_parquet(DATA,columns=columns)
print('Project root:',ROOT,'; study month:',f'{YEAR}-{MONTH:02d}')

## Build daily raw-detection calendar

Calendar cells with no source rows are deliberately labeled `no_rows_coverage_unknown`. The FIRMS hotspot CSVs are detection records, not sensor-coverage logs, so a blank/no-row date is not evidence of zero fire activity.

In [ ]:
start=pd.Timestamp(year=YEAR,month=MONTH,day=1,tz='UTC')
end=start+pd.offsets.MonthBegin(1)-pd.Timedelta(days=1)
calendar=build_daily_activity_calendar(observations,start,end)
summary=summarize_calendar(calendar)
display(summary.sort_values(['product_code','satellite','stream_type']))
display(calendar.groupby('activity_state').size().rename('source-date cells').reset_index())

## Visualize daily detection presence

Color indicates the number of delivered observations. Gray cells mean this source/date has no rows in the supplied detection extract; the files do not tell us whether the sensor had valid coverage, so coverage remains unknown.

In [ ]:
calendar=calendar.copy()
calendar['series']=calendar.product_code+' · '+calendar.satellite+' · '+calendar.stream_type
by_series=(calendar.groupby(['date_utc','series'],as_index=False)
            .agg(detection_count=('detection_count','sum'),has_rows=('has_observation_rows','any')))
values=by_series.pivot(index='date_utc',columns='series',values='detection_count').sort_index()
presence=by_series.pivot(index='date_utc',columns='series',values='has_rows').reindex_like(values)
values=values.where(presence)
cmap=plt.get_cmap('inferno').copy(); cmap.set_bad('#555b61')
fig,ax=plt.subplots(figsize=(13,5))
im=ax.imshow(np.ma.masked_invalid(values.T.to_numpy(dtype=float)),aspect='auto',interpolation='nearest',cmap=cmap)
ax.set_yticks(range(len(values.columns))); ax.set_yticklabels(values.columns)
ax.set_xticks(range(0,len(values.index),max(1,len(values.index)//10)))
ax.set_xticklabels([values.index[i].strftime('%b %d') for i in range(0,len(values.index),max(1,len(values.index)//10))],rotation=45,ha='right')
ax.set_title(f'Daily FIRMS detections by source — {YEAR}-{MONTH:02d} UTC')
ax.set_xlabel('Date (gray = no rows in supplied extract; coverage unknown)'); fig.colorbar(im,ax=ax,label='Detection rows')
plt.tight_layout(); plt.show()

## Descriptive same-month comparison

This compares observation counts and the number of dates with rows. It does not control for observation opportunities, source-stream processing state, clouds, or sensor differences, so it is not an anomaly score.

In [ ]:
comparison=[]
for year in (2025,2026):
    month_start=pd.Timestamp(year=year,month=MONTH,day=1,tz='UTC')
    month_end=month_start+pd.offsets.MonthBegin(1)-pd.Timedelta(days=1)
    c=build_daily_activity_calendar(observations,month_start,month_end)
    s=summarize_calendar(c); s.insert(0,'year',year); comparison.append(s)
comparison_df=pd.concat(comparison,ignore_index=True)
display(comparison_df.sort_values(['product_code','satellite','year']))
print('Interpret as source-specific delivered counts; 2025 archive and 2026 archive/NRT are different processing states.')

## Anomaly-model readiness

This workspace contains one complete calendar year of archive records plus a partial 2026 record set. That is not enough independent seasonal history to fit and validate a stable month-of-year anomaly baseline. Do not flag 2026 as an anomaly from this comparison. Add more consistently processed historical years and, ideally, independent observation-coverage information before enabling anomaly alerts.

## Stage decisions

Keep raw-detection calendars and any later inferred-event calendar as separate views. Treat missing-row dates as unknown coverage. Label same-month comparisons as descriptive. Defer anomaly flags until baseline history and validation are adequate.